In [1]:
data_dir = '~/JEPA/datasets'
output_dir = '~/JEPA/wavjepa_output'

## Visualization functions

In [2]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE
# Optional UMAP (graceful fallback to t-SNE if not installed)
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False


def plot_latent_tsne(
    embeddings,
    labels,
    class_names=None,
    save_path=None,
    wandb_run=None,
    method="tsne",
    title="Latent Space",
):
    """
    Visualize embeddings in 2D using t-SNE (default) or UMAP.

    Subsamples to 5000 points if dataset is large (t-SNE is O(N^2)).

    Args:
        embeddings:   np.ndarray (N, D)
        labels:       np.ndarray (N,) integer class indices
        class_names:  list of class name strings
        save_path:    optional PNG save path
        wandb_run:    optional wandb run object
        method:       "tsne" or "umap" (requires umap-learn)
        title:        plot title

    Returns:
        matplotlib Figure
    """
    if class_names is None:
        class_names = [str(i) for i in range(len(np.unique(labels)))]

    # Subsample if too many points (t-SNE is slow for N > 5000)
    max_samples = 5000
    if len(embeddings) > max_samples:
        idx = np.random.RandomState(42).choice(len(embeddings), max_samples, replace=False)
        embeddings = embeddings[idx]
        labels = labels[idx]

    # Dimensionality reduction
    if method == "umap" and UMAP_AVAILABLE:
        reducer = umap.UMAP(n_components=2, random_state=42, n_jobs=1)
        reduced = reducer.fit_transform(embeddings)
        method_label = "UMAP"
    else:
        if method == "umap":
            print("[vis] umap-learn not installed, falling back to t-SNE.")
        tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
        reduced = tsne.fit_transform(embeddings)
        method_label = "t-SNE"

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.tab10(np.linspace(0, 1, len(class_names)))

    for i, class_name in enumerate(class_names):
        mask = labels == i
        ax.scatter(
            reduced[mask, 0], reduced[mask, 1],
            c=[colors[i]], label=class_name, alpha=0.6, s=10,
        )

    ax.legend(markerscale=2, fontsize=9, loc="best")
    ax.set_title(f"{title} — {method_label}")
    ax.set_xlabel(f"{method_label}-1")
    ax.set_ylabel(f"{method_label}-2")
    fig.tight_layout()

    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

    if wandb_run is not None:
        import wandb
        key = f"vis/{title.lower().replace(' ', '_').replace('/', '_')}_{method_label.lower()}"
        wandb_run.log({key: wandb.Image(fig)}, commit=False)

    return fig

In [3]:
"""
Visualization utilities for image_jepa (image/CIFAR-10 specific).

Designed to work both:
  - From the training loop in main.py (called periodically during training)
  - From a Jupyter/Colab notebook (import and call directly after loading a checkpoint)

All functions accept standard PyTorch objects and optionally log to wandb.

Usage from notebook:
    from examples.image_jepa.vis import visualization_loop
    figs = visualization_loop(model, linear_probe, val_loader, device,
                               save_dir="visualizations", wandb_run=None)
"""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.amp import autocast

# sklearn: required for t-SNE and confusion matrix
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

# Optional UMAP (graceful fallback to t-SNE if not installed)
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False

CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]


# ---------------------------------------------------------------------------
# Internal helpers: dynamic layer probe and naming
# ---------------------------------------------------------------------------

from dataclasses import dataclass
from typing import Dict, List, Optional


@dataclass
class _LayerInfo:
    dot_path: str
    class_name: str       # module.__class__.__name__
    output_shape: tuple   # captured from dummy forward pass
    is_spatial: bool      # True when output is 4-D (N, C, H, W)


def _resolve_module(model, dot_path):
    """Resolve a dot-path string to a submodule within model."""
    module = model
    for attr in dot_path.split("."):
        module = getattr(module, attr)
    return module


def _probe_model(model, device, input_shape=(2, 3, 32, 32)):
    """
    Single dummy forward pass that records the output shape of every named
    module.  Returns an ordered dict ``{dot_path: _LayerInfo}``.

    Works for any backbone (ResNet, ViT, custom) — no hardcoding needed.
    """
    results: Dict[str, _LayerInfo] = {}
    handles: List = []

    def _make_hook(name, cls_name):
        def _h(mod, inp, out):
            if isinstance(out, torch.Tensor) and name not in results:
                results[name] = _LayerInfo(
                    dot_path=name,
                    class_name=cls_name,
                    output_shape=tuple(out.shape),
                    is_spatial=out.dim() == 4,
                )
        return _h

    for name, mod in model.named_modules():
        if name:
            handles.append(mod.register_forward_hook(_make_hook(name, mod.__class__.__name__)))

    dummy = torch.zeros(*input_shape, device=device)
    with torch.no_grad():
        try:
            model(dummy)
        except Exception:
            pass

    for h in handles:
        h.remove()
    return results


def _naming_source_to_key(dot_path: str, class_name: str, idx: Optional[int] = None) -> str:
    """
    Filename/dict-key–safe string encoding both path and class.

    When *idx* is provided (the module's forward-order position in the probe
    dict) it is prepended as a two-digit zero-padded prefix so filenames sort
    in forward order.

    Examples (no idx):
        "backbone.backbone.layer4" + "Sequential" → "backbone_backbone_layer4_Sequential"
    Examples (idx=6):
        "backbone.backbone.layer4" + "Sequential" → "06_backbone_backbone_layer4_Sequential"
    """
    base = f"{dot_path.replace('.', '_')}_{class_name}"
    return f"{idx:02d}_{base}" if idx is not None else base


def _naming_source_to_display(dot_path: str, class_name: str, output_shape: tuple, idx: Optional[int] = None) -> str:
    """
    Human-readable label used in plot titles.

    When *idx* is provided it is prepended as "[06] " so titles identify
    which layer in the forward-pass order is being shown.

    Examples (no idx):
        "backbone.backbone.layer4" shape (2,512,5,5) → "backbone.backbone.layer4 [Sequential | 512×5×5 → GAP'd]"
    Examples (idx=6):
        "backbone.backbone.layer4" shape (2,512,5,5) → "[06] backbone.backbone.layer4 [Sequential | 512×5×5 → GAP'd]"
    """
    if len(output_shape) == 4:
        _, c, h, w = output_shape
        detail = f"{class_name} | {c}\u00d7{h}\u00d7{w} \u2192 GAP'd"
    else:
        d = output_shape[-1]
        detail = f"{class_name} | {d}-d"
    prefix = f"[{idx:02d}] " if idx is not None else ""
    return f"{prefix}{dot_path} [{detail}]"


def _has_sub_children(path: str, probe: Dict[str, "_LayerInfo"]) -> bool:
    """Return True if *path* has any descendant paths in *probe*."""
    prefix = path + "."
    return any(p.startswith(prefix) for p in probe)


def _meaningful_backbone_layers(probe: Dict[str, "_LayerInfo"]) -> List[str]:
    """
    Return block-level backbone outputs: the shallowest depth inside
    ``backbone.*`` that has ≥ N "interesting" modules, where interesting
    means the module is a container (has sub-children) or has "pool" in its
    class name.

    Falls back from min_count 3 → 2 → 1 to handle small / shallow models.

    Works for ResNet, ViT (torchvision), V-JEPA 2, WavJEPA, and any custom
    backbone — no hardcoding of specific attribute paths required.
    """
    backbone_paths = [p for p in probe if p.startswith("backbone.")]
    if not backbone_paths:
        return []

    def _is_interesting(path: str) -> bool:
        cls = probe[path].class_name.lower()
        return "pool" in cls or _has_sub_children(path, probe)

    depths = sorted({len(p.split(".")) for p in backbone_paths})
    min_depth, max_depth = depths[0], depths[-1]

    for min_count in (3, 2, 1):
        for depth in range(min_depth, max_depth + 1):
            at_depth = [p for p in backbone_paths if len(p.split(".")) == depth]
            interesting = [p for p in at_depth if _is_interesting(p)]
            if len(interesting) >= min_count:
                return interesting

    return backbone_paths  # absolute last resort


def _meaningful_projector_layers(probe: Dict[str, "_LayerInfo"]) -> List[str]:
    """
    Return only the output of each complete MLP unit in the projector:
      - ReLU outputs  (end of a Linear→BN→ReLU unit)
      - the final Linear layer (no activation after it)

    Skips bare Linear and BatchNorm nodes that are mid-unit.
    Works for any depth MLP projector.
    """
    # All direct children of projector (depth == 2: "projector.<i>")
    proj_keys = [
        p for p in probe
        if p.startswith("projector.")
        and len(p.split(".")) == 2
    ]
    if not proj_keys:
        return []
    last = proj_keys[-1]
    return [
        p for p in proj_keys
        if probe[p].class_name == "ReLU" or p == last
    ]


def _resolve_layers(
    layers,
    model,
    device,
    probe: Optional[Dict[str, "_LayerInfo"]] = None,
) -> List[str]:
    """
    Normalize the *layers* argument to a concrete list of dot-paths.

    Shorthand strings
    -----------------
    ``None``             → last spatial layer (H > 1) in backbone.* (default)
    ``"all"``            → meaningful backbone layers + meaningful projector layers
    ``"backbone"``       → block-level backbone outputs only (conv1, layer1–4, avgpool…)
    ``"projector"``      → projector unit outputs only (ReLU ends + final Linear)
    ``"backbone_full"``  → every submodule under backbone.*
    ``"projector_full"`` → every submodule under projector.*
    ``"<prefix>"``       → arbitrary dot-path prefix filter (fallback)
    list                 → explicit dot-paths, returned as-is
    """
    if probe is None:
        probe = _probe_model(model, device)

    if isinstance(layers, list):
        return list(layers)

    if layers == "all":
        # Meaningful boundaries across the whole model
        result = _meaningful_backbone_layers(probe) + _meaningful_projector_layers(probe)
        return result if result else list(probe.keys())

    if layers == "backbone":
        result = _meaningful_backbone_layers(probe)
        if result:
            return result
        # Fallback for non-standard backbone nesting
        result = [p for p in probe if p.startswith("backbone.") and len(p.split(".")) == 2]
        return result if result else [p for p in probe if p.startswith("backbone.")]

    if layers == "projector":
        result = _meaningful_projector_layers(probe)
        if result:
            return result
        # Fallback: all direct projector children
        return [p for p in probe if p.startswith("projector.") and len(p.split(".")) == 2]

    if layers == "backbone_full":
        return [p for p in probe if p.startswith("backbone.")]

    if layers == "projector_full":
        return [p for p in probe if p.startswith("projector.")]

    if isinstance(layers, str):
        # Arbitrary prefix filter
        prefix = layers if layers.endswith(".") else layers + "."
        matched = [p for p in probe if p.startswith(prefix) or p == layers]
        if matched:
            return matched
        raise ValueError(
            f"layers={layers!r} matched no modules. "
            f"Available top-level namespaces: "
            + str(sorted({p.split(".")[0] for p in probe}))
        )

    # Default (layers is None): last spatial (H > 1) layer in backbone.*
    spatial_hgt1 = [
        p for p, info in probe.items()
        if info.is_spatial
        and info.output_shape[-2] > 1
        and p.startswith("backbone.")
    ]
    if spatial_hgt1:
        return [spatial_hgt1[-1]]

    # Fallback: last 4-D layer anywhere in backbone.*
    spatial_any = [p for p, info in probe.items()
                   if info.is_spatial and p.startswith("backbone.")]
    if spatial_any:
        return [spatial_any[-1]]

    # Last resort: very last module
    return [list(probe.keys())[-1]]


@torch.no_grad()
def _extract_all_embeddings(model, linear_probe, val_loader, device, dot_paths, probe, use_amp=True):
    """
    Extract embeddings from all *dot_paths* in a single forward pass per batch.

    Spatial layers (4-D output) are Global-Average-Pooled to (N, C) vectors.
    Linear-probe predictions are derived from the model's primary return value.

    Args:
        dot_paths: list of dot-path strings (from ``_resolve_layers``)
        probe:     pre-computed ``{dot_path: _LayerInfo}`` from ``_probe_model``

    Returns:
        embeddings_dict: ``{dot_path: np.ndarray (N, D)}``
        preds:           ``np.ndarray (N,)`` predicted class indices
        targets:         ``np.ndarray (N,)`` ground-truth class indices
    """
    device = torch.device(device) if isinstance(device, str) else device
    model.eval()
    linear_probe.eval()

    buf: Dict[str, List] = {p: [] for p in dot_paths}
    all_preds, all_targets = [], []

    for data, target in val_loader:
        data   = data.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)

        batch_buf = {p: None for p in dot_paths}  # filled by hooks before use
        handles = []
        for p in dot_paths:
            mod = _resolve_module(model, p)
            def _make_hook(path):
                def _h(m, i, o):
                    batch_buf[path] = o.detach().cpu().float()
                return _h
            handles.append(mod.register_forward_hook(_make_hook(p)))

        with autocast(device.type, enabled=use_amp):
            backbone_features, _ = model(data)

        for h in handles:
            h.remove()

        for p in dot_paths:
            t = batch_buf[p]
            assert t is not None, f"Hook for '{p}' did not fire"
            if probe[p].is_spatial:
                buf[p].append(t.mean(dim=(2, 3)))   # GAP → (N, C)
            else:
                buf[p].append(t)                     # already (N, D)

        logits = linear_probe(backbone_features.float())
        all_preds.append(logits.argmax(dim=1).cpu())
        all_targets.append(target.cpu())

    preds   = torch.cat(all_preds,   dim=0).numpy()
    targets = torch.cat(all_targets, dim=0).numpy()
    embeddings_dict = {p: torch.cat(buf[p], dim=0).numpy() for p in dot_paths}

    return embeddings_dict, preds, targets


# ---------------------------------------------------------------------------
# Vis 1: Latent Space Visualization (t-SNE / UMAP)
# ---------------------------------------------------------------------------

def plot_latent_tsne(
    embeddings,
    labels,
    class_names=None,
    save_path=None,
    wandb_run=None,
    method="tsne",
    title="Latent Space",
):
    """
    Visualize embeddings in 2D using t-SNE (default) or UMAP.

    Subsamples to 5000 points if dataset is large (t-SNE is O(N^2)).

    Args:
        embeddings:   np.ndarray (N, D)
        labels:       np.ndarray (N,) integer class indices
        class_names:  list of class name strings
        save_path:    optional PNG save path
        wandb_run:    optional wandb run object
        method:       "tsne" or "umap" (requires umap-learn)
        title:        plot title

    Returns:
        matplotlib Figure
    """
    if class_names is None:
        class_names = [str(i) for i in range(len(np.unique(labels)))]

    # Subsample if too many points (t-SNE is slow for N > 5000)
    max_samples = 5000
    if len(embeddings) > max_samples:
        idx = np.random.RandomState(42).choice(len(embeddings), max_samples, replace=False)
        embeddings = embeddings[idx]
        labels = labels[idx]

    # Dimensionality reduction
    if method == "umap" and UMAP_AVAILABLE:
        reducer = umap.UMAP(n_components=2, random_state=42, n_jobs=1)
        reduced = reducer.fit_transform(embeddings)
        method_label = "UMAP"
    else:
        if method == "umap":
            print("[vis] umap-learn not installed, falling back to t-SNE.")
        tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
        reduced = tsne.fit_transform(embeddings)
        method_label = "t-SNE"

    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.tab10(np.linspace(0, 1, len(class_names)))

    for i, class_name in enumerate(class_names):
        mask = labels == i
        ax.scatter(
            reduced[mask, 0], reduced[mask, 1],
            c=[colors[i]], label=class_name, alpha=0.6, s=10,
        )

    ax.legend(markerscale=2, fontsize=9, loc="best")
    ax.set_title(f"{title} — {method_label}")
    ax.set_xlabel(f"{method_label}-1")
    ax.set_ylabel(f"{method_label}-2")
    fig.tight_layout()

    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

    if wandb_run is not None:
        import wandb
        key = f"vis/{title.lower().replace(' ', '_').replace('/', '_')}_{method_label.lower()}"
        wandb_run.log({key: wandb.Image(fig)}, commit=False)

    return fig


# ---------------------------------------------------------------------------
# Vis 2: Confusion Matrix
# ---------------------------------------------------------------------------

def plot_confusion_matrix(
    preds,
    targets,
    class_names=None,
    save_path=None,
    wandb_run=None,
    normalize=True,
    title="Confusion Matrix",
):
    """
    Plot a confusion matrix from precomputed predictions and targets.

    Args:
        preds:        np.ndarray (N,) predicted class indices
        targets:      np.ndarray (N,) ground-truth class indices
        class_names:  list of class name strings
        save_path:    optional PNG save path
        wandb_run:    optional wandb run object
        normalize:    if True, normalize rows to show per-class recall
        title:        plot title

    Returns:
        matplotlib Figure
    """
    if class_names is None:
        class_names = [str(i) for i in range(len(np.unique(targets)))]

    cm = confusion_matrix(targets, preds, normalize="true" if normalize else None)

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)

    ax.set(
        xticks=np.arange(len(class_names)),
        yticks=np.arange(len(class_names)),
        xticklabels=class_names,
        yticklabels=class_names,
        title=title,
        ylabel="True label",
        xlabel="Predicted label",
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    fmt = ".2f" if normalize else "d"
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, format(cm[i, j], fmt),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=7,
            )

    fig.tight_layout()

    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

    if wandb_run is not None:
        import wandb
        key = f"vis/{title.lower().replace(' ', '_').replace('/', '_')}"
        wandb_run.log({key: wandb.Image(fig)}, commit=False)

    return fig


# ---------------------------------------------------------------------------
# Vis 3: Activation Maps (channel-averaged feature maps via forward hook)
# ---------------------------------------------------------------------------

def plot_activation_maps(
    model,
    val_loader,
    device,
    num_samples=8,
    layer_name="backbone.backbone.layer4",
    save_path=None,
    wandb_run=None,
    title="Activation Maps",
    use_amp=True,
):
    """
    Visualize spatial activation heatmaps from an intermediate ResNet layer,
    overlaid on the input image.

    Uses a forward hook to capture the last spatial feature map, averages
    across channels (simple saliency), and overlays the result as a
    jet-colormap heatmap on top of the original input image.

    Args:
        model:        ImageSSL model (ResNet backbone)
        val_loader:   validation DataLoader
        device:       torch device
        num_samples:  number of images to visualize (first N from first batch)
        layer_name:   dot-path to the backbone layer to hook
                      (default: "backbone.backbone.layer4" for ResNet-18)
        save_path:    optional PNG save path
        wandb_run:    optional wandb run object
        title:        plot title
        use_amp:      enable mixed precision for the forward pass

    Returns:
        matplotlib Figure
    """
    device = torch.device(device) if isinstance(device, str) else device
    model.eval()
    activations = {}

    def _hook(module, input, output):
        # output: (N, C, H', W')
        activations["feat"] = output.detach().cpu().float()

    # Resolve the layer by dot-path
    layer = model
    for attr in layer_name.split("."):
        layer = getattr(layer, attr)
    handle = layer.register_forward_hook(_hook)

    # One forward pass on the first batch
    data, _ = next(iter(val_loader))
    data = data[:num_samples].to(device)

    with torch.no_grad():
        with autocast(device.type, enabled=use_amp):
            model(data)

    handle.remove()

    feat_maps = activations["feat"]           # (N, C, H', W')
    saliency = feat_maps.mean(dim=1)          # (N, H', W') — average across channels

    # Normalize each saliency map independently to [0, 1]
    mins = saliency.flatten(1).min(dim=1).values[:, None, None]
    maxs = saliency.flatten(1).max(dim=1).values[:, None, None]
    saliency = (saliency - mins) / (maxs - mins + 1e-8)

    # Un-normalize CIFAR-10 images for display
    # (mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616])
    data_cpu = data.cpu().float()
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
    std  = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)
    imgs = (data_cpu * std + mean).clamp(0, 1)  # (N, 3, H, W)

    # Plot: row 0 = input images, row 1 = activation overlay
    fig, axes = plt.subplots(2, num_samples, figsize=(num_samples * 2, 4))
    axes = np.array(axes)

    for i in range(num_samples):
        img_np = imgs[i].permute(1, 2, 0).numpy()      # (H, W, 3)
        sal_np = saliency[i].numpy()                    # (H', W')

        # Upsample saliency map to match input image size
        sal_up = F.interpolate(
            torch.tensor(sal_np)[None, None],
            size=img_np.shape[:2],
            mode="bilinear",
            align_corners=False,
        )[0, 0].numpy()

        # Row 0: original input
        axes[0, i].imshow(img_np)
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Input", fontsize=8)

        # Row 1: heatmap overlay
        axes[1, i].imshow(img_np)
        axes[1, i].imshow(sal_up, cmap="jet", alpha=0.5)
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Activations", fontsize=8)

    fig.suptitle(title, fontsize=10)
    fig.tight_layout()

    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

    if wandb_run is not None:
        import wandb
        key = f"vis/{title.lower().replace(' ', '_').replace('/', '_')}"
        wandb_run.log({key: wandb.Image(fig)})

    return fig


# ---------------------------------------------------------------------------
# Main entry point: visualization_loop
# ---------------------------------------------------------------------------

def visualization_loop(
    model,
    linear_probe,
    val_loader,
    device,
    class_names=None,
    use_amp=True,
    save_dir=None,
    wandb_run=None,
    epoch=None,
    tsne_method="tsne",   # backward-compat single-method param
    layers=None,          # None→auto, "all", or list of dot-paths
    tsne_methods=None,    # None→[tsne_method], or ["tsne", "umap"]
):
    """
    Run all image_jepa visualizations.

    Default behaviour (``layers=None``, ``tsne_methods=None``) is identical to
    before: one t-SNE plot, one confusion matrix, one activation map.
    Returned dict keys are also unchanged: ``"tsne"``, ``"confusion"``,
    ``"activation"``.

    Advanced usage from a notebook::

        figs = visualization_loop(
            ...,
            layers=["backbone.backbone.layer2",
                    "backbone.backbone.layer4",
                    "projector.6"],
            tsne_methods=["tsne", "umap"],
        )
        # or auto-discover every named module:
        figs = visualization_loop(..., layers="all")

    When multiple layers / methods are requested the dict keys encode what was
    used: ``"tsne_backbone_backbone_layer4_Sequential_tsne"``,
    ``"activation_backbone_backbone_layer2_Sequential"``, etc.

    Every requested layer gets a t-SNE plot.  Activation maps are generated
    automatically for any layer whose output is 4-D *and* whose spatial
    dimensions are larger than 1×1 (i.e., not post-GAP).

    Args:
        model:        ImageSSL model (or any model whose forward returns
                      ``(backbone_features, projections)``)
        linear_probe: linear classifier used for confusion-matrix predictions
        val_loader:   validation DataLoader
        device:       torch device
        class_names:  list of class name strings (default: CIFAR-10)
        use_amp:      enable automatic mixed precision
        save_dir:     directory for PNG output (``None`` = no file output)
        wandb_run:    wandb run object (``None`` = no wandb logging)
        epoch:        current epoch number (appended to filenames / keys)
        tsne_method:  ``"tsne"`` or ``"umap"`` — backward-compat single param
        layers:       dot-path(s) to visualize.
                      ``None``              → auto (last backbone spatial layer)
                      ``"all"``             → meaningful backbone + projector layers
                      ``"backbone"``        → block-level backbone outputs (layer1–4, etc.)
                      ``"projector"``       → projector unit outputs (ReLU ends + final)
                      ``"backbone_full"``   → every backbone.* submodule
                      ``"projector_full"``  → every projector.* submodule
                      ``"<prefix>"``        → arbitrary dot-path prefix filter
                      list                  → explicit dot-paths
        tsne_methods: list of reduction methods — overrides ``tsne_method``

    Returns:
        dict of ``{key: matplotlib.figure.Figure}``
    """
    if class_names is None:
        class_names = CIFAR10_CLASSES

    methods  = list(tsne_methods) if tsne_methods is not None else [tsne_method]
    suffix   = f"_epoch{epoch:04d}" if epoch is not None else ""
    save_dir = Path(save_dir) if save_dir is not None else None

    # Probe the model once to learn every named module's output shape/type
    probe = _probe_model(model, device)

    # Resolve which dot-paths to extract
    dot_paths = _resolve_layers(layers, model, device, probe=probe)

    # Validate: explicit paths may contain typos not present in probe
    missing = [p for p in dot_paths if p not in probe]
    if missing:
        raise ValueError(
            f"The following dot-paths were not found in the model: {missing}\n"
            f"Available top-level namespaces: "
            + str(sorted({p.split('.')[0] for p in probe}))
        )

    # Forward-order global index: position of each path in the probe dict.
    # Used to prefix filenames/titles so they sort in model-execution order.
    probe_index = {p: i for i, p in enumerate(probe.keys())}

    # single_mode → backward-compat short keys ("tsne", "activation"), no idx prefix
    single_mode = len(dot_paths) == 1 and len(methods) == 1

    # Single pass over val_loader: embeddings for all layers + preds + targets
    embeddings_dict, preds, targets = _extract_all_embeddings(
        model, linear_probe, val_loader, device, dot_paths, probe, use_amp=use_amp
    )

    figs = {}
    close_after = wandb_run is not None or save_dir is not None

    # --- Vis 1: Latent space (t-SNE / UMAP) — one plot per (layer, method) ---
    for dot_path in dot_paths:
        info = probe[dot_path]
        _idx = probe_index[dot_path]
        key  = _naming_source_to_key(dot_path, info.class_name, idx=None if single_mode else _idx)
        disp = _naming_source_to_display(dot_path, info.class_name, info.output_shape,
                                         idx=None if single_mode else _idx)

        for method in methods:
            if single_mode:
                fig_key = "tsne"
                fname   = f"latent_{method}{suffix}.png"
                title   = f"Latent Space{suffix}"
            else:
                fig_key = f"tsne_{key}_{method}"
                fname   = f"latent_{key}_{method}{suffix}.png"
                title   = f"Latent Space — {disp}{suffix}"

            fig = plot_latent_tsne(
                embeddings_dict[dot_path], targets,
                class_names=class_names,
                save_path=str(save_dir / fname) if save_dir else None,
                wandb_run=wandb_run,
                method=method,
                title=title,
            )
            figs[fig_key] = fig
            if close_after:
                plt.close(fig)

    # --- Vis 2: Confusion matrix (always one) ---
    fig = plot_confusion_matrix(
        preds, targets,
        class_names=class_names,
        save_path=str(save_dir / f"confusion_matrix{suffix}.png") if save_dir else None,
        wandb_run=wandb_run,
        title=f"Confusion Matrix{suffix}",
    )
    figs["confusion"] = fig
    if close_after:
        plt.close(fig)

    # --- Vis 3: Activation maps — spatial (4-D, H > 1) layers only ---
    for dot_path in dot_paths:
        info = probe[dot_path]
        # Skip flat or post-GAP (1×1) layers — activation maps are meaningless
        if not info.is_spatial:
            continue
        _, _, h, w = info.output_shape
        if h <= 1 and w <= 1:
            continue

        _idx = probe_index[dot_path]
        key  = _naming_source_to_key(dot_path, info.class_name, idx=None if single_mode else _idx)
        disp = _naming_source_to_display(dot_path, info.class_name, info.output_shape,
                                         idx=None if single_mode else _idx)

        if single_mode:
            fig_key = "activation"
            fname   = f"activation_maps{suffix}.png"
            title   = f"Activation Maps{suffix}"
        else:
            fig_key = f"activation_{key}"
            fname   = f"activation_{key}{suffix}.png"
            title   = f"Activation Maps — {disp}{suffix}"

        fig = plot_activation_maps(
            model, val_loader, device,
            layer_name=dot_path,
            save_path=str(save_dir / fname) if save_dir else None,
            wandb_run=wandb_run,
            title=title,
            use_amp=use_amp,
        )
        figs[fig_key] = fig
        if close_after:
            plt.close(fig)

    return figs


# ---------------------------------------------------------------------------
# Checkpoint-based entry point (for notebook / ad-hoc use)
# ---------------------------------------------------------------------------

def visualize_from_checkpoint(
    ckpt_path,
    cfg_path=None,
    save_dir=None,
    wandb_run=None,
    tsne_method="tsne",      # backward-compat single-method param
    # --- Advanced options; passed directly to visualization_loop ---
    layers=None,             # None→auto, "all", or list of dot-paths
    tsne_methods=None,       # None→[tsne_method], or ["tsne", "umap"]
):
    """
    Load a checkpoint and run all visualizations in one call.

    ``cfg_path`` is optional — if not provided, ``config.yaml`` is
    auto-discovered next to the checkpoint file (saved there automatically
    during training).

    ``save_dir`` defaults to ``<exp_dir>/visualizations/<checkpoint_stem>``,
    e.g.::

        .../resnet_bcs_seed42/visualizations/latest/
        .../resnet_bcs_seed42/visualizations/epoch_0020/

    Simple usage (same as always)::

        figs = visualize_from_checkpoint(ckpt_path=".../latest.pth.tar")
        figs["tsne"].show()
        figs["confusion"].show()
        figs["activation"].show()

    Advanced usage — multi-layer, multi-method, inline display::

        figs = visualize_from_checkpoint(
            ckpt_path=".../latest.pth.tar",
            save_dir=None,
            layers=[
                "backbone.backbone.layer2",
                "backbone.backbone.layer4",
                "projector.6",
            ],
            tsne_methods=["tsne", "umap"],
        )
        figs["tsne_backbone_backbone_layer4_Sequential_tsne"].show()
        figs["activation_backbone_backbone_layer2_Sequential"].show()

        # Namespace shorthands:
        figs = visualize_from_checkpoint(ckpt_path="...", layers="all")             # meaningful backbone + projector
        figs = visualize_from_checkpoint(ckpt_path="...", layers="backbone")        # block-level backbone only
        figs = visualize_from_checkpoint(ckpt_path="...", layers="projector")       # projector unit outputs only
        figs = visualize_from_checkpoint(ckpt_path="...", layers="backbone_full")   # every backbone.* submodule
        figs = visualize_from_checkpoint(ckpt_path="...", layers="projector_full")  # every projector.* submodule
    """
    # Lazy imports — use model.py to avoid pulling in training-only deps (fire, wandb...)
    from examples.image_jepa.model import ImageSSL, ResNet18
    from examples.image_jepa.eval import LinearProbe
    from examples.image_jepa.dataset import get_val_transforms
    from eb_jepa.training_utils import load_config, load_checkpoint
    from torchvision.datasets import CIFAR10
    from torch.utils.data import DataLoader
    import os

    ckpt_path = Path(ckpt_path)

    # Default save_dir: <exp_dir>/visualizations/<checkpoint_stem>
    # Checkpoints are always saved as .pth.tar → strip both extensions. e.g. latest.pth.tar → latest, epoch_0020.pth.tar → epoch_0020
    if save_dir is None:
        ckpt_stem = Path(ckpt_path.stem).stem  # strips .tar then .pth
        save_dir = ckpt_path.parent / "visualizations" / ckpt_stem

    # Auto-discover config.yaml next to checkpoint if not provided
    if cfg_path is None:
        cfg_path = ckpt_path.parent / "config.yaml"
        if not cfg_path.exists():
            raise FileNotFoundError(
                f"No config.yaml found next to checkpoint at {ckpt_path.parent}.\n"
                "Either provide cfg_path explicitly or re-run training to save config.yaml."
            )

    # 1. Load config
    cfg = load_config(cfg_path)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 2. Build model and linear probe
    from examples.image_jepa.model import build_model, build_linear_probe
    model, features_dim = build_model(cfg)
    model = model.to(device)

    # 3. Build linear probe
    linear_probe = build_linear_probe(features_dim).to(device)

    # 4. Load checkpoint weights
    ckpt_info = load_checkpoint(ckpt_path, model, optimizer=None, device=device)
    if "linear_probe_state_dict" in ckpt_info:
        linear_probe.load_state_dict(ckpt_info["linear_probe_state_dict"])

    # Derive epoch from the checkpoint filename (e.g. "epoch_10.pth.tar" → 10)
    # to avoid the off-by-one that arises when main.py increments epoch before
    # saving (so the stored value is always 1 more than the filename suggests).
    # Fall back to the stored value only if the filename has no trailing digits.
    import re
    _stem = Path(ckpt_path).stem                       # "epoch_10.pth" or "epoch_10"
    _stem = re.sub(r"\.[^.]+$", "", _stem)             # strip inner ext: "epoch_10.pth.tar" → stem is already "epoch_10"
    _m = re.search(r"(\d+)$", _stem)
    epoch = int(_m.group(1)) if _m else ckpt_info.get("epoch", 0)
    print(f"Loaded checkpoint from epoch {epoch}")

    # 5. Build val loader
    data_dir = os.environ.get("EBJEPA_DSETS", ".")
    val_dataset = CIFAR10(root=data_dir, train=False, download=True,
                          transform=get_val_transforms())
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.data.batch_size,
        shuffle=False,
        num_workers=cfg.data.num_workers,
        pin_memory=True,
    )

    # 6. Run visualizations
    return visualization_loop(
        model=model,
        linear_probe=linear_probe,
        val_loader=val_loader,
        device=device,
        save_dir=save_dir,
        wandb_run=wandb_run,
        epoch=epoch,
        tsne_method=tsne_method,
        layers=layers,
        tsne_methods=tsne_methods,
    )


## Setup

In [26]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install -U transformers

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 12.0 MB 18.4 MB/s            
     |████████████████████████████████| 507 kB 105.0 MB/s            
     |████████████████████████████████| 566 kB 112.0 MB/s            
     |████████████████████████████████| 791 kB 107.1 MB/s            
     |████████████████████████████████| 3.3 MB 83.8 MB/s            
     |████████████████████████████████| 78 kB 18.1 MB/s            
     |████████████████████████████████| 200 kB 86.1 MB/s            
     |████████████████████████████████| 4.2 MB 131.4 MB/s            
  Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)


## Download and Extract ESC-50

Download the ESC-50 dataset from the specified GitHub URL and extract it into a local directory using shell commands.



In [5]:
import os
import zipfile
import requests
import shutil

# 1. Check for the zip in data_dir or current dir before downloading
url = 'https://github.com/karolpiczak/ESC-50/archive/master.zip'
zip_filename = 'esc50_master.zip'
possible_paths = [os.path.join(data_dir, zip_filename), zip_filename]

zip_path = next((p for p in possible_paths if os.path.exists(p)), None)

if not zip_path:
    zip_path = zip_filename
    print(f'Downloading {url}...')
    r = requests.get(url, stream=True)
    with open(zip_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
else:
    print(f'Found existing zip at: {zip_path}')

# 2. Simplified Extraction
if os.path.exists('esc50'): shutil.rmtree('esc50')
if os.path.exists('ESC-50-master'): shutil.rmtree('ESC-50-master')

print('Extracting archive...')
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('.')

# The zip contains a single folder 'ESC-50-master', rename it to 'esc50'
os.rename('ESC-50-master', 'esc50')

# 3. Verify
print('\nVerifying extraction:')
audio_dir = os.path.join('esc50', 'audio')
meta_file = os.path.join('esc50', 'meta', 'esc50.csv')

if os.path.isdir(audio_dir) and os.path.isfile(meta_file):
    print('Success: Audio folder and metadata CSV are present.')
else:
    print('Failure: Missing expected folders or files.')

Extracting archive...

Verifying extraction:
Success: Audio folder and metadata CSV are present.


## Create ESC-50 and ESC-10 file+label lists

In [6]:
import pandas as pd
import os

# 1. Load the full ESC-50 metadata
esc50_df = pd.read_csv(meta_file)
esc50_file_paths = [os.path.join(audio_dir, fname) for fname in esc50_df['filename']]
esc50_labels = esc50_df['target'].values
esc50_class_names = sorted(esc50_df['category'].unique())

# 2. Filter for ESC-10 subset
esc10_df = esc50_df[esc50_df['esc10'] == True].reset_index(drop=True)
esc10_file_paths = [os.path.join(audio_dir, fname) for fname in esc10_df['filename']]
esc10_labels = esc10_df['target'].values
esc10_class_names = sorted(esc10_df['category'].unique())

# 3. Show statistics for ESC-50
print("--- FULL ESC-50 DATASET ---")
print(f"Total samples: {len(esc50_df)}")
print(f"Unique classes ({len(esc50_class_names)}): {esc50_class_names}")
print("\nSamples per class (first 5):")
print(esc50_df['category'].value_counts().head(5))
display(esc50_df.head())

# 4. Show statistics for ESC-10
print("\n--- ESC-10 SUBSET ---")
print(f"Total samples: {len(esc10_df)}")
print(f"Unique classes ({len(esc10_class_names)}): {esc10_class_names}")
print("\nSamples per class:")
print(esc10_df['category'].value_counts())
display(esc10_df.head())

# Set default pointers for the next extraction steps to esc10 as per previous task flow
file_paths = esc10_file_paths
labels = esc10_labels
class_names_list = esc10_class_names

--- FULL ESC-50 DATASET ---
Total samples: 2000
Unique classes (50): ['airplane', 'breathing', 'brushing_teeth', 'can_opening', 'car_horn', 'cat', 'chainsaw', 'chirping_birds', 'church_bells', 'clapping', 'clock_alarm', 'clock_tick', 'coughing', 'cow', 'crackling_fire', 'crickets', 'crow', 'crying_baby', 'dog', 'door_wood_creaks', 'door_wood_knock', 'drinking_sipping', 'engine', 'fireworks', 'footsteps', 'frog', 'glass_breaking', 'hand_saw', 'helicopter', 'hen', 'insects', 'keyboard_typing', 'laughing', 'mouse_click', 'pig', 'pouring_water', 'rain', 'rooster', 'sea_waves', 'sheep', 'siren', 'sneezing', 'snoring', 'thunderstorm', 'toilet_flush', 'train', 'vacuum_cleaner', 'washing_machine', 'water_drops', 'wind']

Samples per class (first 5):
category
dog                40
chirping_birds     40
vacuum_cleaner     40
thunderstorm       40
door_wood_knock    40
Name: count, dtype: int64


,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-100038-A-14.wav,1,14,chirping_birds,False,100038,A
2,1-100210-A-36.wav,1,36,vacuum_cleaner,False,100210,A
3,1-100210-B-36.wav,1,36,vacuum_cleaner,False,100210,B
4,1-101296-A-19.wav,1,19,thunderstorm,False,101296,A



--- ESC-10 SUBSET ---
Total samples: 400
Unique classes (10): ['chainsaw', 'clock_tick', 'crackling_fire', 'crying_baby', 'dog', 'helicopter', 'rain', 'rooster', 'sea_waves', 'sneezing']

Samples per class:
category
dog               40
chainsaw          40
crackling_fire    40
helicopter        40
rain              40
crying_baby       40
clock_tick        40
sneezing          40
rooster           40
sea_waves         40
Name: count, dtype: int64


,filename,fold,target,category,esc10,src_file,take
0,1-100032-A-0.wav,1,0,dog,True,100032,A
1,1-110389-A-0.wav,1,0,dog,True,110389,A
2,1-116765-A-41.wav,1,41,chainsaw,True,116765,A
3,1-17150-A-12.wav,1,12,crackling_fire,True,17150,A
4,1-172649-A-40.wav,1,40,helicopter,True,172649,A


## Find meaninful layer list of WAVJEPA


In [8]:
! pip install torchaudio

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 4.0 MB 20.5 MB/s            
     |████████████████████████████████| 888.0 MB 19 kB/s               
     |████████████████████████████████| 267.5 MB 237.5 MB/s            
     |████████████████████████████████| 288.2 MB 251.5 MB/s            
     |████████████████████████████████| 954 kB 106.0 MB/s            
     |████████████████████████████████| 6.3 MB 52.8 MB/s            
     |████████████████████████████████| 594.3 MB 164 kB/s              
     |████████████████████████████████| 193.1 MB 176 kB/s              
     |████████████████████████████████| 322.4 MB 248.8 MB/s            
     |████████████████████████████████| 1.6 MB 108.0 MB/s            
     |████████████████████████████████| 88.0 MB 112.5 MB/s            
     |████████████████████████████████| 287.2 MB 247.3 MB/s            
     |████████████████████████████████| 1.2 MB 111.8 MB/s            

In [15]:
import torch
import torchaudio
import numpy as np
from transformers import AutoModel, AutoFeatureExtractor, AutoConfig
from tqdm.auto import tqdm

# 1. Initialize config to get the class
model_id = "labhamlet/wavjepa-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model {model_id} to {device}...")

config = AutoConfig.from_pretrained(model_id, trust_remote_code=True)

# 2. Patch the class specifically from the auto-factory mapping before instantiation
from transformers.models.auto.auto_factory import _get_model_class
model_class = _get_model_class(config, AutoModel._model_mapping)

# Define the missing property on the class itself
if not hasattr(model_class, 'all_tied_weights_keys'):
    model_class.all_tied_weights_keys = property(lambda self: {})

# 3. Now load model and feature extractor
model = AutoModel.from_pretrained(model_id, config=config, trust_remote_code=True).to(device)
model.eval()
print("Model loaded and patched successfully.")

Loading model labhamlet/wavjepa-base to cuda...


KeyError: 'WavJEPAConfig'

## List wavjepa layers

In [ ]:
import torch

# 1. Prepare dummy input for 10 seconds of 16kHz audio: (batch, channels, samples)
dummy_input_shape = (1, 1, 160000)

# 2. Probe the model architecture using the helper function
# This captures every named module that fires during the forward pass
print(f"Probing WavJEPA architecture with input shape {dummy_input_shape}...")
wavjepa_full_probe = _probe_model(model, device, input_shape=dummy_input_shape)

# 3. Print the chronological list of all layers
print(f"\nDetected {len(wavjepa_full_probe)} modules in the execution path:\n")
print("Index | Layer Path / Name")
print("-" * 60)
for i, path in enumerate(wavjepa_full_probe.keys()):
    info = wavjepa_full_probe[path]
    print(f"[{i:03d}] | {path} ({info.class_name})")

Probing WavJEPA architecture with input shape (1, 1, 160000)...

Detected 139 modules in the execution path:

Index | Layer Path / Name
------------------------------------------------------------
[000] | model.extract_audio.cnn.0.0 (Conv1d)
[001] | model.extract_audio.cnn.0.1 (Dropout)
[002] | model.extract_audio.cnn.0.2 (GroupNorm)
[003] | model.extract_audio.cnn.0.3 (GELU)
[004] | model.extract_audio.cnn.0 (Sequential)
[005] | model.extract_audio.cnn.1.0 (Conv1d)
[006] | model.extract_audio.cnn.1.1 (Dropout)
[007] | model.extract_audio.cnn.1.2 (GELU)
[008] | model.extract_audio.cnn.1 (Sequential)
[009] | model.extract_audio.cnn.2.0 (Conv1d)
[010] | model.extract_audio.cnn.2.1 (Dropout)
[011] | model.extract_audio.cnn.2.2 (GELU)
[012] | model.extract_audio.cnn.2 (Sequential)
[013] | model.extract_audio.cnn.3.0 (Conv1d)
[014] | model.extract_audio.cnn.3.1 (Dropout)
[015] | model.extract_audio.cnn.3.2 (GELU)
[016] | model.extract_audio.cnn.3 (Sequential)
[017] | model.extract_audio.cnn

/usr/local/lib/python3.12/dist-packages/torch/masked/maskedtensor/creation.py:20: UserWarning: The PyTorch API of MaskedTensors is in prototype stage and will change in the near future. Please open a Github issue for features requests and see our documentation on the torch.masked module for further information about the project.
  return MaskedTensor(data, mask, requires_grad)


In [ ]:
# 1. Define the logical cutoff points
print(f"--- WavJEPA Architecture Summary ---")
print(f"[000-026] Part 1: Audio Feature Extractor (CNN)")
print(f"[027-028] Transition: Mapping to Transformer Dimension")
print(f"[029-137] Part 2: Transformer Encoder (12 Layers + Norm)")
print(f"[138]       Part 3: Final Encoder Container Output")

# 2. Select meaningful layers for visualization (shallow to deep)
visualization_candidates = []

# The final output of the CNN extractor
visualization_candidates.append('model.extract_audio')

# Include ALL Transformer blocks (0 through 11) to see fine-grained progression
for i in range(12):
    visualization_candidates.append(f'model.encoder.layers.{i}')

# The final normalization output
visualization_candidates.append('model.encoder.norm')

# The absolute final output of the Encoder module container
visualization_candidates.append('model.encoder')

print(f"\nUpdated layers for t-SNE (Shallow to Deep):")
for i, path in enumerate(visualization_candidates):
    info = wavjepa_full_probe[path]
    print(f"[{i}] Index {list(wavjepa_full_probe.keys()).index(path):03d} | {path} ({info.class_name})")

# Save this list for our next visualization task
selected_dot_paths = visualization_candidates

--- WavJEPA Architecture Summary ---
[000-026] Part 1: Audio Feature Extractor (CNN)
[027-028] Transition: Mapping to Transformer Dimension
[029-137] Part 2: Transformer Encoder (12 Layers + Norm)
[138]       Part 3: Final Encoder Container Output

Updated layers for t-SNE (Shallow to Deep):
[0] Index 026 | model.extract_audio (ConvFeatureExtractor)
[1] Index 037 | model.encoder.layers.0 (TransformerEncoderLayer)
[2] Index 046 | model.encoder.layers.1 (TransformerEncoderLayer)
[3] Index 055 | model.encoder.layers.2 (TransformerEncoderLayer)
[4] Index 064 | model.encoder.layers.3 (TransformerEncoderLayer)
[5] Index 073 | model.encoder.layers.4 (TransformerEncoderLayer)
[6] Index 082 | model.encoder.layers.5 (TransformerEncoderLayer)
[7] Index 091 | model.encoder.layers.6 (TransformerEncoderLayer)
[8] Index 100 | model.encoder.layers.7 (TransformerEncoderLayer)
[9] Index 109 | model.encoder.layers.8 (TransformerEncoderLayer)
[10] Index 118 | model.encoder.layers.9 (TransformerEncoderLaye

In [ ]:
import os
import torch
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import torchaudio
from transformers import AutoFeatureExtractor

def extract_and_visualize_latent_progression(
    model,
    extractor,
    file_paths,
    labels,
    class_names,
    selected_layers,
    device,
    save_base_dir,
    prefix=""
):
    """
    Extracts features from multiple layers and saves t-SNE plots for each.
    """
    os.makedirs(save_base_dir, exist_ok=True)
    layer_embeddings = {path: [] for path in selected_layers}

    print(f"[Extracted] {prefix} | Layers: {len(selected_layers)} | Files: {len(file_paths)}")

    # 1. Feature Extraction Loop
    for path in tqdm(file_paths, desc=f"Extracting {prefix}"):
        # Load and preprocess
        audio, sr = torchaudio.load(path)
        if sr != 16000:
            audio = torchaudio.functional.resample(audio, orig_freq=sr, new_freq=16000)
        if audio.shape[0] > 1:
            audio = torch.mean(audio, dim=0, keepdim=True)

        inputs = extractor(audio, sampling_rate=16000, return_tensors="pt").to(device)

        current_activations = {}
        handles = []

        def get_activation(name):
            def hook(mod, inp, out):
                act = out[0] if isinstance(out, tuple) else out
                current_activations[name] = act.detach().cpu()
            return hook

        for layer_path in selected_layers:
            module = dict(model.named_modules())[layer_path]
            handles.append(module.register_forward_hook(get_activation(layer_path)))

        with torch.no_grad():
            model(inputs['input_values'])

        for layer_path in selected_layers:
            act = current_activations[layer_path]
            if act.ndim == 3:
                pooled = act.mean(dim=1).numpy().squeeze()
            else:
                pooled = act.numpy().squeeze()
            layer_embeddings[layer_path].append(pooled)

        for h in handles:
            h.remove()

    # 2. Visualization Loop
    print(f"[Visualizing] {prefix}...")
    for i, layer_path in enumerate(selected_layers):
        embeddings_array = np.vstack(layer_embeddings[layer_path])

        fig = plot_latent_tsne(
            embeddings=embeddings_array,
            labels=labels,
            class_names=class_names,
            method="tsne",
            title=f"WavJEPA {prefix} - Layer {i}: {layer_path}"
        )

        fname = f"{prefix}_layer_{i:02d}_{layer_path.replace('.', '_')}.png"
        save_path = os.path.join(save_base_dir, fname)
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)

    print(f"[Done] {prefix} plots saved to: {save_base_dir}")

# --- Execution ---

# Initialize Extractor
extractor = AutoFeatureExtractor.from_pretrained(model_id, trust_remote_code=True)
base_vis_dir = '/content/drive/MyDrive/Colab Notebooks/JEPA/WAVJEPA/visualizations'

# 1. Run for ESC-10
esc10_target_to_idx = {name: i for i, name in enumerate(esc10_class_names)}
esc10_mapped_labels = np.array([esc10_target_to_idx[cat] for cat in esc10_df['category']])

extract_and_visualize_latent_progression(
    model=model,
    extractor=extractor,
    file_paths=esc10_file_paths,
    labels=esc10_mapped_labels,
    class_names=esc10_class_names,
    selected_layers=selected_dot_paths,
    device=device,
    save_base_dir=os.path.join(base_vis_dir, 'esc10'),
    prefix="ESC10"
)

# 2. Run for ESC-50
extract_and_visualize_latent_progression(
    model=model,
    extractor=extractor,
    file_paths=esc50_file_paths,
    labels=esc50_labels,
    class_names=esc50_class_names,
    selected_layers=selected_dot_paths,
    device=device,
    save_base_dir=os.path.join(base_vis_dir, 'esc50'),
    prefix="ESC50"
)

preprocessor_config.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

feature_extraction_wavjepa.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/labhamlet/wavjepa-base:
- feature_extraction_wavjepa.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


[Extracted] ESC10 | Layers: 15 | Files: 400


Extracting ESC10:   0%|          | 0/400 [00:00<?, ?it/s]

/root/.cache/huggingface/modules/transformers_modules/labhamlet/wavjepa_hyphen_base/6be4a5093f6c9adbbd55e00b6e5b8f067aa03345/feature_extraction_wavjepa.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  audio = torch.tensor(audio)
/usr/local/lib/python3.12/dist-packages/torch/masked/maskedtensor/creation.py:20: UserWarning: The PyTorch API of MaskedTensors is in prototype stage and will change in the near future. Please open a Github issue for features requests and see our documentation on the torch.masked module for further information about the project.
  return MaskedTensor(data, mask, requires_grad)


[Visualizing] ESC10...
[Done] ESC10 plots saved to: /content/drive/MyDrive/Colab Notebooks/JEPA/WAVJEPA/visualizations/esc10
[Extracted] ESC50 | Layers: 15 | Files: 2000


Extracting ESC50:   0%|          | 0/2000 [00:00<?, ?it/s]

/root/.cache/huggingface/modules/transformers_modules/labhamlet/wavjepa_hyphen_base/6be4a5093f6c9adbbd55e00b6e5b8f067aa03345/feature_extraction_wavjepa.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  audio = torch.tensor(audio)
/usr/local/lib/python3.12/dist-packages/torch/masked/maskedtensor/creation.py:20: UserWarning: The PyTorch API of MaskedTensors is in prototype stage and will change in the near future. Please open a Github issue for features requests and see our documentation on the torch.masked module for further information about the project.
  return MaskedTensor(data, mask, requires_grad)


[Visualizing] ESC50...
[Done] ESC50 plots saved to: /content/drive/MyDrive/Colab Notebooks/JEPA/WAVJEPA/visualizations/esc50


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# 1. Select a small subset (e.g., ESC-10) for a quick test
subset_paths = esc10_file_paths
subset_labels = esc10_mapped_labels

mean_pooled_features = []
cls_token_features = []

print(f"Comparing Mean Pooling vs. CLS Token for {len(subset_paths)} files...")

for path in tqdm(subset_paths):
    audio, sr = torchaudio.load(path)
    if sr != 16000:
        audio = torchaudio.functional.resample(audio, orig_freq=sr, new_freq=16000)
    if audio.shape[0] > 1:
        audio = torch.mean(audio, dim=0, keepdim=True)

    inputs = extractor(audio, sampling_rate=16000, return_tensors="pt").to(device)

    with torch.no_grad():
        # Get the final hidden states
        # For most HF models, outputs[0] is the last hidden state: (Batch, Seq, Hidden)
        outputs = model(inputs['input_values'])
        last_hidden_state = outputs[0]

        # Method A: Mean Pooling (what we used)
        mean_p = last_hidden_state.mean(dim=1).cpu().numpy().squeeze()
        mean_pooled_features.append(mean_p)

        # Method B: CLS Token (The first token [0] in the sequence)
        cls_p = last_hidden_state[:, 0, :].cpu().numpy().squeeze()
        cls_token_features.append(cls_p)

# 2. Visualize both to find the 'missing' quality
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

print("Generating Comparison t-SNE...")
plot_latent_tsne(np.vstack(mean_pooled_features), subset_labels, class_names=esc10_class_names, title="Method A: Mean Pooled (Current)")
plt.show()

plot_latent_tsne(np.vstack(cls_token_features), subset_labels, class_names=esc10_class_names, title="Method B: CLS Token (Potential Fix)")
plt.show()